In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# --- Load your Excel file ---
file_path = "Data.xlsx"
df = pd.read_excel(file_path)

# --- Thin lens prediction ---
def v_pred(u, f):
    return (f * u) / (u - f)

# --- Chi-squared function ---
def chi_squared(f, u, v_obs, sigma=1.0):
    v_calc = v_pred(u, f)
    residuals = (v_obs - v_calc) / sigma
    return np.sum(residuals**2)

# Column index pairs for the six measurement blocks (zero-based)
col_pairs = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12)]

for i, (col_u, col_v) in enumerate(col_pairs, start=1):
    # Extract rows 3–5 (Trials 1–3)
    u = df.iloc[2:5, col_u].astype(float).to_numpy()
    v_meas = df.iloc[2:5, col_v].astype(float).to_numpy()

    # Skip if any missing data
    if np.isnan(u).any() or np.isnan(v_meas).any():
        print(f"Block {i}: Skipped (missing data)")
        continue

    # Minimize chi-squared to find best f
    res = minimize(lambda f: chi_squared(f, u, v_meas), x0=[50])
    best_f = res.x[0]
    chi2_min = res.fun
    red_chi2 = chi2_min / (len(u) - 1)

    # Print results
    print(f"\nBlock {i}:")
    print(f"  Best-fit focal length f = {best_f:.3f} mm")
    print(f"  Minimum chi-squared     = {chi2_min:.3f}")
    print(f"  Reduced chi-squared     = {red_chi2:.3f}")


Block 1:
  Best-fit focal length f = 18.989 mm
  Minimum chi-squared     = 1.671
  Reduced chi-squared     = 0.836

Block 2:
  Best-fit focal length f = 11.902 mm
  Minimum chi-squared     = 3.627
  Reduced chi-squared     = 1.814

Block 3:
  Best-fit focal length f = 12.249 mm
  Minimum chi-squared     = 6.932
  Reduced chi-squared     = 3.466

Block 4:
  Best-fit focal length f = 32.382 mm
  Minimum chi-squared     = 12.083
  Reduced chi-squared     = 6.042

Block 5:
  Best-fit focal length f = 14.632 mm
  Minimum chi-squared     = 23.639
  Reduced chi-squared     = 11.820
Block 6: Skipped (missing data)
